In [17]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter
from sklearn.naive_bayes import MultinomialNB

In [1]:
# 1. 訓練資料
train_data = [
    ("Hi, lunch today?", "ham"),
    ("WINNER! Claim your cash prize now!", "spam"),
    ("Call me back later.", "ham"),
    ("IMPORTANT: Your account security issue, click here.", "spam"),
    ("Free entry to lucky draw, text WIN.", "spam"),
    ("Meeting at 5pm to discuss project.", "ham")
]

## 自訂一class MultinomialNB_0
#### fit method 用來訓練(生成式模型)：
        1. CountVectorizer 特徵提取器將訓練樣本轉成(fit_transform)數值矩陣(csr matrix)，每個數字代表各詞彙發生在各樣本的詞頻
        2. 依樣本計算各類別的事前機率
        3. 依類別計算在各詞彙發生次數
        4. 依類別計算總詞彙數
        5. 依類別計算各詞彙對數機率
#### predict用來判別歸屬類別(判別式模型)：
        1. 測試文本樣本使用前面經過訓練的特徵提取器轉成(transform)數值矩陣
        2. 樣本出現的詞彙對應於訓練集對應的詞彙對數機率，依據公式計算各類別的機率
        3. 並比較大小判別其應屬類別
#### predict_proba 用來計算歸屬類別的機率(判別式模型)：
        1. 同predict 1.
        2. 同predict 2.

In [ ]:
class MultinomialNB_0:
    def __init__(self):
        self.vectorizer = CountVectorizer(token_pattern=r'(?u)\b\w+\b')  # 確保標點符號不影響      
    def __proba_score(self,X):  # 斷詞
        test_X = self.vectorizer.transform([X]).toarray()[0]
        active_word_indices = np.where(test_X > 0)[0]
        word_counts = test_X[active_word_indices]
        proba_score=np.exp(self.class_log_prior_+np.sum(word_counts*self.feature_log_prob_[:,active_word_indices],axis=1))
        return proba_score      
    # 2. 訓練階段：各類別的詞彙對數機率 (計算多項分布的MLE)    
    def fit(self, data):
        alpha = 1
        X = self.vectorizer.fit_transform([text for text,label in data]) # X part
        y = [label for text,label in data]                          # y part
        self.classes_, counts = np.unique(y, return_counts=True)
        self.class_log_prior_ = np.log(counts/counts.sum())     # 各類別的事前機率
        class_indices = [[np.where(np.array(y)==x)[0] for x in cls] for cls in [self.classes_]][0]
        self.vocabulary_= self.vectorizer.vocabulary_    # 詞彙表(詞彙，索引)
        self.n_features_in_ = X.shape[1]    # 相異詞彙總個數
        # self.feature_count_ 各類別在各詞彙發生次數
        self.feature_count_=[X.astype(float).toarray()[indices,:].sum(axis=0) for indices in class_indices]
        # self.feature_count_alpha_ laplace smoothing 給予分子初始值alpha
        self.feature_count_alpha_ = np.zeros((len(self.classes_),self.n_features_in_))+alpha        
        # total_words 各類別詞彙數(含重複出現各算)
        total_words = np.array([X.toarray()[indices,:].sum(axis=1)
                                for indices in class_indices]).sum(axis=1)
        print(total_words)
        # self.feature_log_prob_ 各類別的詞彙對數機率
        self.feature_log_prob_ = np.log(((self.feature_count_+self.feature_count_alpha_)/
                                         (total_words[:,np.newaxis]+self.n_features_in_)))
    # 3. 預測類別
    def predict(self, X):
        proba_score = self.__proba_score(X)
        return self.classes_[np.argmax(proba_score)]
    # 4. 預測類別機率
    def predict_proba(self,X):
        proba_score = self.__proba_score(X)
        return [proba_score/np.sum(proba_score)]

# 使用自訂模型MultinomialNB_0
#### 1. 訓練資料投入訓練
#### 2. 測試資料進行預測(predict)
#### 3. 測試資料預測類別機率(predict_proba)

In [285]:
model = MultinomialNB_0()
model.fit(train_data)

test_email = "Congratulations! You won a free prize, click here."
print(f'預測結果： {model.predict(test_email)}')
probabilities=model.predict_proba(test_email)
for prob in probabilities:
    print(f"正常機率 :{prob[0]}    垃圾機率 :{prob[1]}")

[13 20]
預測結果： spam
正常機率 :0.10137450605372134    垃圾機率 :0.8986254939462788


# 使用同樣的訓練資料改用sklearn.naive_bayes.MultinomialNB
        1. CountVectorizer.fit_transform特徵提取訓練資料轉成數值矩陣
        2. MultinomialNB.fit完成訓練        

In [288]:
from sklearn.naive_bayes import MultinomialNB

In [289]:
clf = MultinomialNB()
vectorizer = CountVectorizer()
X = np.array(train_data)[:,0]       # convert list to array
X = vectorizer.fit_transform(X)     # csr matrix BOW
y = np.array(train_data)[:,1]
clf.fit(X, y)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


# 使用同樣的測試資料
        1. 運用已完成訓練的MultinomialNB模型進行測試資料的predict
        2. 與自訂MultinomialNB_0模型比較類別機率是否一致

In [287]:
new_X = vectorizer.transform([test_email])
predictions = clf.predict(new_X)
print(predictions)
sklearn_probas=clf.predict_proba(new_X)
for prob in sklearn_probas:
    print(f"正常機率 :{prob[0]}    垃圾機率 :{prob[1]}")
print(f"兩者是否完全吻合？ : {np.allclose(sklearn_probas[0], probabilities)}")

['spam']
正常機率 :0.10137450605372132    垃圾機率 :0.8986254939462787
兩者是否完全吻合？ : True
